In [ ]:
# Instalando o pacote com bibliotecas groq
!pip install groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.7/139.7 kB 2.0 MB/s eta 0:00:00


In [ ]:
# Definindo minha chave de API do groq
import os
os.environ['GROQ_API_KEY'] = "Minha chave de API"

In [ ]:
# Teste para verificar a chave de API chamando a IA via chat_completions
import os
from groq import Groq

client = Groq(
    api_key=os.environ.get("GROQ_API_KEY"),
)

chat_completion = client.chat.completions.create(
    messages=[
        {
            "role": "user",
            "content": "Explain the importance of fast language models",
        }
    ],
    model="llama-3.3-70b-versatile",
)

print(chat_completion.choices[0].message.content)

Fast language models are crucial in the field of natural language processing (NLP) and have numerous applications in various industries. The importance of fast language models can be understood from the following perspectives:

1. **Real-time Processing**: Fast language models enable real-time processing of text data, which is essential for applications such as chatbots, virtual assistants, and language translation. Quick processing allows for instant responses, improving user experience and engagement.
2. **Large-Scale Deployment**: Fast language models can handle large volumes of text data, making them ideal for deployment in industries such as social media, customer service, and online forums. This enables companies to analyze and respond to customer feedback, monitor trends, and detect potential issues efficiently.
3. **Reduced Latency**: Fast language models minimize latency, which is critical for applications that require rapid response times, such as:
	* Sentiment analysis: Quic

In [ ]:
class Agent: # Molde para criar um agente
  def __init__(self, client, system): # Funcao construtora: guarda o cliente API, as instrucoes iniciais do agente e cria o historico de conversas do agente
    self.client = client
    self.system = system
    self.messages = []
    if self.system is not None:
      self.messages.append({"role": "system", "content": self.system})

  def __call__(self, message=""): # Executa o objeto agente como funcao adicionando uma mensagem do usuario ao historico de mensagens
    if message:
      self.messages.append({"role": "user", "content": message})
    result = self.execute()
    self.messages.append({"role": "assistant", "content": result})
    return result

  def execute(self): # Envia o historico de mensagens ao groq e devolve a resposta da LLM
    completion = client.chat.completions.create(
        messages=self.messages,
        model="llama-3.3-70b-versatile",
    )
    return completion.choices[0].message.content

In [ ]:
system_prompt = """
You run in a loop of Thought, Action, PAUSE, Observation.
At the end of the loop you output an Answer
Use Thought to describe your thoughts about the question you have been asked.
Use Action to run one of the actions available to you - then return PAUSE.
Observation will be the result of running those actions.

Your available actions are:

calculate:
e.g. calculate: 4 * 7 / 3
Runs a calculation and returns the number - uses Python so be sure to use floating point syntax if necessary

get_planet_mass:
e.g. get_planet_mass: Earth
returns weight of the planet in kg

Example session:

Question: What is the mass of Earth times 2?
Thought: I need to find the mass of Earth
Action: get_planet_mass: Earth
PAUSE

You will be called again with this:

Observation: 5.972e24

Thought: I need to multiply this by 2
Action: calculate: 5.972e24 * 2
PAUSE

You will be called again with this:

Observation: 1,1944×10e25

If you have the answer, output it as the Answer.

Answer: The mass of Earth times 2 is 1,1944×10e25.

Now it's your turn:
""".strip() # Prompt contendo as instrucoes de como a IA deve interpretar o historico de mensagens e responder (Como o agente deve agir)

In [ ]:
#ferramentas que o agente utilizara em sua atuacao (dados planetarios e a operacao passada pelo usuario)
def calculate(operation):
  return eval(operation)

def get_planet_mass(planet) -> float:
    match planet.lower():
        case "earth":
            return 5.972e24
        case "jupiter":
            return 1.898e27
        case "mars":
            return 6.39e23
        case "mercury":
            return 3.285e23
        case "neptune":
            return 1.024e26
        case "saturn":
            return 5.683e26
        case "uranus":
            return 8.681e25
        case "venus":
            return 4.867e24
        case _:
            return 0.0

In [ ]:
neil_tyson = Agent(client, system_prompt) # Inicializacao de um agente para execucao sem loop

In [ ]:
result = neil_tyson("What is the mass of Earth times 5?")
print(result)

Thought: To find the mass of Earth times 5, I first need to determine the mass of Earth. 
Action: get_planet_mass: Earth
PAUSE


In [ ]:
neil_tyson.messages


[{'role': 'system',
  'content': "You run in a loop of Thought, Action, PAUSE, Observation.\nAt the end of the loop you output an Answer\nUse Thought to describe your thoughts about the question you have been asked.\nUse Action to run one of the actions available to you - then return PAUSE.\nObservation will be the result of running those actions.\n\nYour available actions are:\n\ncalculate:\ne.g. calculate: 4 * 7 / 3\nRuns a calculation and returns the number - uses Python so be sure to use floating point syntax if necessary\n\nget_planet_mass:\ne.g. get_planet_mass: Earth\nreturns weight of the planet in kg\n\nExample session:\n\nQuestion: What is the mass of Earth times 2?\nThought: I need to find the mass of Earth\nAction: get_planet_mass: Earth\nPAUSE\n\nYou will be called again with this:\n\nObservation: 5.972e24\n\nThought: I need to multiply this by 2\nAction: calculate: 5.972e24 * 2\nPAUSE\n\nYou will be called again with this:\n\nObservation: 1,1944×10e25\n\nIf you have the a

In [ ]:
result = neil_tyson()
print(result)

In [ ]:
observation = get_planet_mass("Earth")
print(observation)

5.972e+24


In [ ]:
next_prompt = f"Observation: {observation}"
result = neil_tyson(next_prompt)
print(result)

Thought: Now that I have the mass of Earth, I need to multiply this value by 5 to find the result.
Action: calculate: 5.972e24 * 5
PAUSE


In [ ]:
neil_tyson.messages

[{'role': 'system',
  'content': "You run in a loop of Thought, Action, PAUSE, Observation.\nAt the end of the loop you output an Answer\nUse Thought to describe your thoughts about the question you have been asked.\nUse Action to run one of the actions available to you - then return PAUSE.\nObservation will be the result of running those actions.\n\nYour available actions are:\n\ncalculate:\ne.g. calculate: 4 * 7 / 3\nRuns a calculation and returns the number - uses Python so be sure to use floating point syntax if necessary\n\nget_planet_mass:\ne.g. get_planet_mass: Earth\nreturns weight of the planet in kg\n\nExample session:\n\nQuestion: What is the mass of Earth times 2?\nThought: I need to find the mass of Earth\nAction: get_planet_mass: Earth\nPAUSE\n\nYou will be called again with this:\n\nObservation: 5.972e24\n\nThought: I need to multiply this by 2\nAction: calculate: 5.972e24 * 2\nPAUSE\n\nYou will be called again with this:\n\nObservation: 1,1944×10e25\n\nIf you have the a

In [ ]:
observation = calculate("3.285e23 * 5")
print(observation)

1.6425e+24


In [ ]:
next_prompt = f"Observation: {observation}"
result = neil_tyson(next_prompt)
print(result)

Thought: I have now calculated the mass of Earth times 5, so I can provide the final answer.
Answer: The mass of Earth times 5 is 2.986e+25.


In [ ]:
import re

def agent_loop(max_interations, system, query):
  agent = Agent(client, system_prompt) # inicializando o agente
  tools = ['calculate', 'get_planet_mass'] # lista com as ferramentas que a IA tem permissão para usar
  next_prompt = query
  i = 0
  while i < max_interations: # Loop para manter a conversa fluindo entre o código e a IA
    i += 1
    result = agent(next_prompt)
    print(result)

    if "PAUSE" in result and "Action" in result:
      action = re.findall(r"Action: ([a-z_]+): (.+)", result, re.IGNORECASE)
      chosen_tool = action[0][0]
      arg = action[0][1]

      if chosen_tool in tools:
        result_tool = eval(f"{chosen_tool}('{arg}')")
        next_prompt = f"Observation: {result_tool}"
      # A cada volta, ele manda a mensagem atual para a IA com result = agent(next_prompt) e imprime o que a IA pensou

      else:
        next_prompt = "Observation: Tool not found"

      print(next_prompt)
      continue

    if "Answer" in result:
      break

In [ ]:
agent_loop(max_interations=10, system=system_prompt, query="What is the mass of the Earth plus the mass of mercury and all of it times 5?")

Thought: To find the mass of the Earth plus the mass of Mercury and all of it times 5, I first need to find the masses of Earth and Mercury. 

Action: get_planet_mass: Earth
PAUSE
Observation: 5.972e+24
Thought: Now that I have the mass of Earth, I need to find the mass of Mercury.

Action: get_planet_mass: Mercury
PAUSE
Observation: 3.285e+23
Thought: Now that I have the masses of both Earth and Mercury, I can add them together and then multiply the result by 5.

Action: calculate: (5.972e+24 + 3.285e+23) * 5
PAUSE
Observation: 3.1502500000000004e+25
Thought: I have now calculated the mass of Earth plus the mass of Mercury and multiplied the result by 5, so I have all the necessary information to provide the answer.

Answer: The mass of the Earth plus the mass of Mercury and all of it times 5 is 3.15025e+25.
